# FluidX IntelliXcap 96

The FluidX IntelliXcap 96 (an Azenta brand) is an automated screw-cap **decapper**. It unscrews, holds, and screws back on all 96 caps of a screw-cap tube rack in a single stroke. A plate mover loads the rack onto its one nest.

## Resources

- [Azenta IntelliXcap user manual](https://web.azenta.com/hubfs/azenta-files/resources/manuals-guides/319430-IXC-User-Manual.pdf)
- [Azenta IntelliXcap 96 product page](https://www.azenta.com/products/intellixcap-automated-screw-cap-decapper-recapper-96-format)

| Property | Value |
| --- | --- |
| Communication | Serial, STX/ETX-framed ASCII |
| Serial settings | 9600 baud, 8 data bits, no parity, 1 stop bit, no handshake |
| Framing | each reply is wrapped in STX (`0x02`) .. ETX (`0x03`) |
| Reply pattern | ACK (`0x06`), then `<cmd>OK` echo, then a result frame |

Every command is a single character written followed by ETX. A motion command is accepted with an ACK and a `<cmd>OK` echo; the status word then reads `StatusBUSY` while it moves. The terminal state depends on the operation: decap finishes at `StatusRECAP` while the head holds caps, whereas recap and waste finish at `StatusOK`. A refused or no-op command answers `CommandIgnore`.

## Physical setup



Connect the decapper's serial port to your computer through its USB-to-serial adapter and use the stable `by-id` path so the port survives re-enumeration, e.g. `/dev/serial/by-id/usb-FTDI_USB_Serial_Converter_XXXXXXXX-if00-port0`.



Tube volume is not a driver setting. The installed IntelliCartridge and its firmware profile define the supported tube height, cap geometry/thread, and motion settings. Use the cartridge specified for the exact tube family: two nominally 0.5 mL tube types may require different cartridges. Cartridge IDs 1–14 load their matching profiles automatically; extended cartridges require the corresponding stored profile to be selected on the instrument. The Python commands are otherwise identical for every supported tube type. See the [official Azenta IntelliXcap user manual](https://web.azenta.com/hubfs/azenta-files/resources/manuals-guides/319430-IXC-User-Manual.pdf) for cartridge/profile setup and compatibility guidance.



```{important}

If the instrument comes up reporting `StatusBUSY` and ignores commands, it is

locked out. The most common cause is an **engaged e-stop**; the safety guard/hood

and other interlocks do the same. There is no command to read the e-stop

directly, so `setup()` raises with this hint when it sees `StatusBUSY` at connect.

Clear the e-stop and retry.

```

## Connect



`setup()` opens the serial port and reads the status. It raises if the device is not ready (e.g. e-stop).

In [ ]:
from pylabrobot.azenta.fluidx import FluidXIntelliXcap96



decapper = FluidXIntelliXcap96(port="/dev/ttyUSB0")  # replace with your port

await decapper.setup()

## Status



`request_status()` returns the current status word: `StatusOK` (idle with no caps held), `StatusBUSY` (moving), `StatusRECAP` (decapped; caps held and ready to recap or waste), `StatusSLEEP` (standby), or `StatusMANUAL` (the instrument requires inspection and touchscreen recovery). Note the status word does **not** encode the tray position.

In [ ]:
print("status:", await decapper.request_status())

### Error codes

The legacy serial protocol may report a generic operation failure such as `DecapERROR` without including the instrument's numeric error code. When a reply does include a known code, `FluidXError` decodes it automatically and exposes it as `error_code`. If the code is available separately from the instrument error log, look up its documented meaning with `get_error_message()` or construct the same exception with `FluidXError.from_error_code()`.

```python
from pylabrobot.azenta.fluidx import FluidXError, get_error_message

print(get_error_message(145))
# Light curtain calibration max retries exceeded.

error = FluidXError.from_error_code(145)
print(error)
# IntelliXcap error 145: Light curtain calibration max retries exceeded.
```

The complete local mapping comes from the error table in the [official Azenta IntelliXcap user manual](https://web.azenta.com/hubfs/azenta-files/resources/manuals-guides/319430-IXC-User-Manual.pdf).

## Tray

Open and close the loading tray. Asking for a position the tray is already in is a harmless no-op. Tray motion does not change whether caps are held, so after decapping the terminal state remains `StatusRECAP` rather than returning to `StatusOK`.

In [ ]:
await decapper.open_tray()

In [ ]:
await decapper.close_tray()

## Home



Home all axes.

In [ ]:
await decapper.home()

## Decap, recap, waste

With a rack of screw-cap tubes loaded on the nest, `decap()` unscrews and holds all 96 caps. After decapping, choose **one** terminal operation: `recap()` screws those caps back onto the tubes, while `waste()` releases them into a separately positioned cap carrier. `decap()` refuses if caps are already held (recap or waste first); `recap()` refuses if none are held.

A fault (e.g. decapping with no rack loaded) can latch the device in `StatusError`, and the failing call raises a `FluidXError` carrying the firmware's message. A latched error is cleared **only by homing** -- the reset command is ignored on this firmware. By default (`auto_recover=True`) the next operation you issue homes to clear `StatusError` and then proceeds; pass `auto_recover=False` to have it raise instead. `StatusMANUAL` requires an explicit safety decision: inspect the rack and cap head, confirm that axis motion is safe, then call `reset_error()`. Hardware testing confirmed that it homes the machine from `StatusMANUAL` through `StatusBUSY` to `StatusOK`. Normal motion commands remain blocked until recovery completes.

```{warning}
`waste()` is irreversible and does not detect whether a cap carrier is present. Before calling it, open the tray, remove the sample-tube rack, and position the correct cap carrier or collection vessel according to your instrument's procedure. On hardware, leaving the tube rack beneath the head caused the released caps to fall back onto the tube openings. They looked recapped but were not guaranteed to be threaded or torqued. See the [official Azenta IntelliXcap user manual](https://web.azenta.com/hubfs/azenta-files/resources/manuals-guides/319430-IXC-User-Manual.pdf), which defines waste as dropping caps into a carrier.
```

In [ ]:
# After inspecting the rack/head and confirming motion is safe:
await decapper.reset_error()

In [ ]:
await decapper.decap()   # unscrew and hold all 96 caps

In [ ]:
# Choose this path to retain the caps.
await decapper.recap()   # screw and torque the held caps back on

In [ ]:
# Alternative to recap: with caps held, remove the tube rack and position
# the correct cap carrier first. Do not run this after recap.
await decapper.waste()   # irreversibly release held caps into the carrier

## Standby



Put the decapper to sleep with `standby()` and wake it with `ready()`.

In [ ]:
await decapper.standby()

In [ ]:
await decapper.ready()

## Teardown



`stop()` closes the serial connection.

In [ ]:
await decapper.stop()